### Setup

In [26]:
import importlib
import polars as pl

pl.Config.set_tbl_rows(25)

from pathlib import Path

REPO_DIR = Path('.').resolve().parent
DEV_DATA_DIR = REPO_DIR / 'dev-data'

import sys
sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'src' / 'util'))

In [27]:
KID_ID = 'NA12878'
DAD_ID = 'NA12891'
MOM_ID = 'NA12892'

VCF_TRIO_PHASED = str(DEV_DATA_DIR / 'output' / 'CEPH-1463.joint.GRCh38.deepvariant.glnexus.phased.vcf.gz')
OUTPUT_DIR = str(DEV_DATA_DIR / 'output')

def blocks_tsv_path(uid):
    return str(DEV_DATA_DIR / 'output' / f'CEPH-1463.joint.GRCh38.deepvariant.glnexus.phased.{uid}.blocks.tsv')

BLOCKS_TSV_KID = blocks_tsv_path(KID_ID)
BLOCKS_TSV_DAD = blocks_tsv_path(DAD_ID)
BLOCKS_TSV_MOM = blocks_tsv_path(MOM_ID)

print(f'VCF: {VCF_TRIO_PHASED}')
print(f'Output dir: {OUTPUT_DIR}')

VCF: /Users/petermchale/tapestry/dev-data/output/CEPH-1463.joint.GRCh38.deepvariant.glnexus.phased.vcf.gz
Output dir: /Users/petermchale/tapestry/dev-data/output


### Read per-sample het SNV alleles and phase block IDs from the WhatsHap trio-phased VCF

In [28]:
import get_trio_phasing
importlib.reload(get_trio_phasing)
from get_trio_phasing import get_trio_read_phasing

PHASING = get_trio_read_phasing(VCF_TRIO_PHASED, KID_ID, DAD_ID, MOM_ID)
DF_KID = PHASING[KID_ID]
DF_DAD = PHASING[DAD_ID]
DF_MOM = PHASING[MOM_ID]

print(f'Kid het SNVs: {len(DF_KID)}')
print(f'Dad het SNVs: {len(DF_DAD)}')
print(f'Mom het SNVs: {len(DF_MOM)}')

Kid het SNVs: 690
Dad het SNVs: 1059
Mom het SNVs: 801


In [29]:
DF_KID

chrom,start,end,REF,ALT,phase_block_id,allele_hap1,allele_hap2
str,i64,i64,str,str,str,str,str
"""chr1""",1000334,1000335,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1000730,1000731,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1002744,1002745,"""G""","""A""","""1000335""","""1""","""0"""
"""chr1""",1004624,1004625,"""A""","""G""","""1000335""","""1""","""0"""
"""chr1""",1004715,1004716,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1005903,1005904,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1005953,1005954,"""G""","""A""","""1000335""","""1""","""0"""
"""chr1""",1006158,1006159,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1008306,1008307,"""G""","""C""","""1000335""","""1""","""0"""


In [30]:
DF_DAD

chrom,start,end,REF,ALT,phase_block_id,allele_hap1,allele_hap2
str,i64,i64,str,str,str,str,str
"""chr1""",1000334,1000335,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1000730,1000731,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1002744,1002745,"""G""","""A""","""1000335""","""1""","""0"""
"""chr1""",1004624,1004625,"""A""","""G""","""1000335""","""1""","""0"""
"""chr1""",1004715,1004716,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1004822,1004823,"""G""","""A""","""1000335""","""0""","""1"""
"""chr1""",1005903,1005904,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1005953,1005954,"""G""","""A""","""1000335""","""1""","""0"""
"""chr1""",1006158,1006159,"""C""","""T""","""1000335""","""1""","""0"""


In [31]:
DF_MOM

chrom,start,end,REF,ALT,phase_block_id,allele_hap1,allele_hap2
str,i64,i64,str,str,str,str,str
"""chr1""",1000334,1000335,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1000730,1000731,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1002744,1002745,"""G""","""A""","""1000335""","""0""","""1"""
"""chr1""",1004624,1004625,"""A""","""G""","""1000335""","""0""","""1"""
"""chr1""",1004715,1004716,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1005903,1005904,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1005953,1005954,"""G""","""A""","""1000335""","""0""","""1"""
"""chr1""",1006158,1006159,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1008306,1008307,"""G""","""C""","""1000335""","""0""","""1"""


### Get phase blocks for each individual (pre-computed by `run-whatshap.sh`)

In [32]:
import get_trio_phasing
importlib.reload(get_trio_phasing)
from get_trio_phasing import get_trio_phase_blocks

DF_BLOCKS_KID = get_trio_phase_blocks(BLOCKS_TSV_KID, KID_ID)
DF_BLOCKS_DAD = get_trio_phase_blocks(BLOCKS_TSV_DAD, DAD_ID)
DF_BLOCKS_MOM = get_trio_phase_blocks(BLOCKS_TSV_MOM, MOM_ID)

print(f'Kid phase blocks: {len(DF_BLOCKS_KID)}')
print(f'Dad phase blocks: {len(DF_BLOCKS_DAD)}')
print(f'Mom phase blocks: {len(DF_BLOCKS_MOM)}')

Kid phase blocks: 1
Dad phase blocks: 1
Mom phase blocks: 1


In [33]:
DF_BLOCKS_KID

chrom,start,end,phase_block_id,num_variants
str,i64,i64,str,i64
"""chr1""",1000334,1999756,"""1000335""",802


In [34]:
DF_BLOCKS_DAD

chrom,start,end,phase_block_id,num_variants
str,i64,i64,str,i64
"""chr1""",1000334,1999860,"""1000335""",1231


In [35]:
DF_BLOCKS_MOM

chrom,start,end,phase_block_id,num_variants
str,i64,i64,str,i64
"""chr1""",1000334,1993898,"""1000335""",949


### Compute hap-map blocks (intersection of phase blocks across kid, dad, mom)

In [36]:
import get_trio_phasing
importlib.reload(get_trio_phasing)
from get_trio_phasing import get_trio_hap_map_blocks

DF_HAP_MAP_BLOCKS = get_trio_hap_map_blocks(DF_BLOCKS_KID, DF_BLOCKS_DAD, DF_BLOCKS_MOM)
print(f'Hap-map blocks: {len(DF_HAP_MAP_BLOCKS)}')
DF_HAP_MAP_BLOCKS

Hap-map blocks: 1


chrom,start,end
str,i64,i64
"""chr1""",1000334,1993898


### Build the trio haplotype map via bit-vector comparison

Labeling convention:
- **A** = dad's hap1, **B** = dad's hap2 (fixed)
- **C** = mom's hap1, **D** = mom's hap2 (fixed)
- Kid's hap1 (paternal) matches A or B per block; kid's hap2 (maternal) matches C or D per block

Due to meiotic recombination, the kid's paternal haplotype may correspond to A in some blocks and B in others.

In [37]:
import get_trio_hap_map
importlib.reload(get_trio_hap_map)
from get_trio_hap_map import get_trio_hap_map

DF_HAP_MAP = get_trio_hap_map(
    DF_KID, DF_DAD, DF_MOM,
    DF_HAP_MAP_BLOCKS,
    KID_ID, DAD_ID, MOM_ID,
)
print(f'Hap map blocks: {len(DF_HAP_MAP)}')
DF_HAP_MAP

Hap map blocks: 1


chrom,start,end,paternal_haplotype,maternal_haplotype,paternal_concordance,maternal_concordance,num_het_SNVs_pat,num_het_SNVs_mat
str,i64,i64,str,str,f64,f64,i64,i64
"""chr1""",1000334,1993898,"""A""","""C""",1.0,1.0,1,1


In [38]:
print('Paternal haplotype value counts:')
print(DF_HAP_MAP['paternal_haplotype'].value_counts())
print()
print('Maternal haplotype value counts:')
print(DF_HAP_MAP['maternal_haplotype'].value_counts())

Paternal haplotype value counts:
shape: (1, 2)
┌────────────────────┬───────┐
│ paternal_haplotype ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ A                  ┆ 1     │
└────────────────────┴───────┘

Maternal haplotype value counts:
shape: (1, 2)
┌────────────────────┬───────┐
│ maternal_haplotype ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ C                  ┆ 1     │
└────────────────────┴───────┘


In [39]:
print('Blocks with paternal concordance < 1.0:')
DF_HAP_MAP.filter(pl.col('paternal_concordance') < 1.0)

Blocks with paternal concordance < 1.0:


chrom,start,end,paternal_haplotype,maternal_haplotype,paternal_concordance,maternal_concordance,num_het_SNVs_pat,num_het_SNVs_mat
str,i64,i64,str,str,f64,f64,i64,i64


In [40]:
print('Blocks with maternal concordance < 1.0:')
DF_HAP_MAP.filter(pl.col('maternal_concordance') < 1.0)

Blocks with maternal concordance < 1.0:


chrom,start,end,paternal_haplotype,maternal_haplotype,paternal_concordance,maternal_concordance,num_het_SNVs_pat,num_het_SNVs_mat
str,i64,i64,str,str,f64,f64,i64,i64


### Phase methylation to A, B, C, D for all three individuals

This step requires methylation BED files from `aligned_bam_to_cpg_scores` (run on the WhatsHap-haplotagged BAMs).

In [41]:
METH_BASE_DIR = DEV_DATA_DIR / 'output' / 'dna-methylation'

def meth_bed_paths(uid):
    """Return (count_hap1, count_hap2, model_hap1, model_hap2) paths for a sample."""
    count_dir = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.count.trio-phased'
    model_dir = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.model.trio-phased'
    return (
        str(count_dir / f'{uid}.GRCh38.haplotagged.hap1.bed.gz'),
        str(count_dir / f'{uid}.GRCh38.haplotagged.hap2.bed.gz'),
        str(model_dir / f'{uid}.GRCh38.haplotagged.hap1.bed.gz'),
        str(model_dir / f'{uid}.GRCh38.haplotagged.hap2.bed.gz'),
    )

KID_METH = meth_bed_paths(KID_ID)
DAD_METH = meth_bed_paths(DAD_ID)
MOM_METH = meth_bed_paths(MOM_ID)

import os
for label, paths in [('Kid', KID_METH), ('Dad', DAD_METH), ('Mom', MOM_METH)]:
    for p in paths:
        exists = os.path.exists(p)
        print(f'{label}: {Path(p).name} -> {"EXISTS" if exists else "MISSING"}')

Kid: NA12878.GRCh38.haplotagged.hap1.bed.gz -> EXISTS
Kid: NA12878.GRCh38.haplotagged.hap2.bed.gz -> EXISTS
Kid: NA12878.GRCh38.haplotagged.hap1.bed.gz -> EXISTS
Kid: NA12878.GRCh38.haplotagged.hap2.bed.gz -> EXISTS
Dad: NA12891.GRCh38.haplotagged.hap1.bed.gz -> EXISTS
Dad: NA12891.GRCh38.haplotagged.hap2.bed.gz -> EXISTS
Dad: NA12891.GRCh38.haplotagged.hap1.bed.gz -> EXISTS
Dad: NA12891.GRCh38.haplotagged.hap2.bed.gz -> EXISTS
Mom: NA12892.GRCh38.haplotagged.hap1.bed.gz -> EXISTS
Mom: NA12892.GRCh38.haplotagged.hap2.bed.gz -> EXISTS
Mom: NA12892.GRCh38.haplotagged.hap1.bed.gz -> EXISTS
Mom: NA12892.GRCh38.haplotagged.hap2.bed.gz -> EXISTS


In [42]:
import get_meth_hap1_hap2
importlib.reload(get_meth_hap1_hap2)
from get_meth_hap1_hap2 import read_meth_hap1_hap2

DF_METH_COUNT_HAP1_HAP2_KID = read_meth_hap1_hap2(
    pb_cpg_tool_mode='count',
    bed_hap1=KID_METH[0],
    bed_hap2=KID_METH[1],
)
print(f'Kid count-based meth: {len(DF_METH_COUNT_HAP1_HAP2_KID)} rows')
DF_METH_COUNT_HAP1_HAP2_KID

Kid count-based meth: 35280 rows


chrom,start,end,total_read_count_hap1,methylation_level_hap1,total_read_count_hap2,methylation_level_hap2
str,i64,i64,i64,f64,i64,f64
"""chr1""",990862,990863,10,0.7,null,null
"""chr1""",990869,990870,10,0.7,null,null
"""chr1""",990914,990915,10,0.7,null,null
"""chr1""",990924,990925,10,0.7,null,null
"""chr1""",990948,990949,11,0.818,null,null
"""chr1""",990956,990957,11,0.818,null,null
"""chr1""",991280,991281,11,0.818,null,null
"""chr1""",991431,991432,12,0.75,null,null
"""chr1""",991460,991461,12,0.75,null,null


In [43]:
import phase_meth_to_parent_haps
importlib.reload(phase_meth_to_parent_haps)
from phase_meth_to_parent_haps import phase_meth_to_haplotypes

DF_METH_COUNT_PHASED_KID = phase_meth_to_haplotypes(DF_METH_COUNT_HAP1_HAP2_KID, DF_HAP_MAP, 'kid')
print(f'Kid phased meth: {len(DF_METH_COUNT_PHASED_KID)} rows')
DF_METH_COUNT_PHASED_KID.filter(pl.col('start_hap_map_block').is_not_null()).head(10)

Kid phased meth: 35280 rows


chrom,start,end,start_hap_map_block,end_hap_map_block,paternal_haplotype,maternal_haplotype,methylation_level_pat,methylation_level_mat,total_read_count_pat,total_read_count_mat
str,i64,i64,i64,i64,str,str,f64,f64,f64,f64
"""chr1""",1000334,1000335,1000334,1993898,"""A""","""C""",0.0,0.111,26.0,18.0
"""chr1""",1000343,1000344,1000334,1993898,"""A""","""C""",0.154,0.278,26.0,18.0
"""chr1""",1000349,1000350,1000334,1993898,"""A""","""C""",0.231,0.278,26.0,18.0
"""chr1""",1000356,1000357,1000334,1993898,"""A""","""C""",0.077,0.111,26.0,18.0
"""chr1""",1000358,1000359,1000334,1993898,"""A""","""C""",0.115,0.167,26.0,18.0
"""chr1""",1000361,1000362,1000334,1993898,"""A""","""C""",0.077,0.222,26.0,18.0
"""chr1""",1000371,1000372,1000334,1993898,"""A""","""C""",0.154,0.5,26.0,18.0
"""chr1""",1000380,1000381,1000334,1993898,"""A""","""C""",0.154,0.111,26.0,18.0
"""chr1""",1000384,1000385,1000334,1993898,"""A""","""C""",0.154,0.167,26.0,18.0


In [44]:
DF_METH_COUNT_HAP1_HAP2_DAD = read_meth_hap1_hap2(
    pb_cpg_tool_mode='count',
    bed_hap1=DAD_METH[0],
    bed_hap2=DAD_METH[1],
)
DF_METH_COUNT_PHASED_DAD = phase_meth_to_haplotypes(DF_METH_COUNT_HAP1_HAP2_DAD, DF_HAP_MAP, 'dad')
print(f'Dad phased meth: {len(DF_METH_COUNT_PHASED_DAD)} rows')
DF_METH_COUNT_PHASED_DAD.filter(pl.col('start_hap_map_block').is_not_null()).head(10)

Dad phased meth: 34861 rows


chrom,start,end,start_hap_map_block,end_hap_map_block,methylation_level_A,methylation_level_B,total_read_count_A,total_read_count_B
str,i64,i64,i64,i64,f64,f64,f64,f64
"""chr1""",1000334,1000335,1000334,1993898,0.077,0.0,13.0,21.0
"""chr1""",1000343,1000344,1000334,1993898,0.231,0.095,13.0,21.0
"""chr1""",1000349,1000350,1000334,1993898,0.077,0.048,13.0,21.0
"""chr1""",1000356,1000357,1000334,1993898,0.077,0.0,13.0,21.0
"""chr1""",1000358,1000359,1000334,1993898,0.077,0.048,13.0,21.0
"""chr1""",1000361,1000362,1000334,1993898,0.154,0.143,13.0,21.0
"""chr1""",1000371,1000372,1000334,1993898,0.077,0.048,13.0,21.0
"""chr1""",1000380,1000381,1000334,1993898,0.077,0.19,13.0,21.0
"""chr1""",1000384,1000385,1000334,1993898,0.077,0.19,13.0,21.0


In [45]:
DF_METH_COUNT_HAP1_HAP2_MOM = read_meth_hap1_hap2(
    pb_cpg_tool_mode='count',
    bed_hap1=MOM_METH[0],
    bed_hap2=MOM_METH[1],
)
DF_METH_COUNT_PHASED_MOM = phase_meth_to_haplotypes(DF_METH_COUNT_HAP1_HAP2_MOM, DF_HAP_MAP, 'mom')
print(f'Mom phased meth: {len(DF_METH_COUNT_PHASED_MOM)} rows')
DF_METH_COUNT_PHASED_MOM.filter(pl.col('start_hap_map_block').is_not_null()).head(10)

Mom phased meth: 33325 rows


chrom,start,end,start_hap_map_block,end_hap_map_block,methylation_level_C,methylation_level_D,total_read_count_C,total_read_count_D
str,i64,i64,i64,i64,f64,f64,f64,f64
"""chr1""",1000334,1000335,1000334,1993898,0.0,0.0,19.0,29.0
"""chr1""",1000343,1000344,1000334,1993898,0.211,0.207,19.0,29.0
"""chr1""",1000349,1000350,1000334,1993898,0.105,0.103,19.0,29.0
"""chr1""",1000356,1000357,1000334,1993898,0.053,0.107,19.0,28.0
"""chr1""",1000358,1000359,1000334,1993898,0.053,0.143,19.0,28.0
"""chr1""",1000361,1000362,1000334,1993898,0.105,0.071,19.0,28.0
"""chr1""",1000371,1000372,1000334,1993898,0.105,0.25,19.0,28.0
"""chr1""",1000380,1000381,1000334,1993898,0.211,0.107,19.0,28.0
"""chr1""",1000384,1000385,1000334,1993898,0.211,0.107,19.0,28.0


### Notes

- The dev data covers only a ~1Mb region of chr1, so the hap map will be small.
- Methylation phasing cells require running `aligned_bam_to_cpg_scores` on the WhatsHap-haplotagged BAMs in `dev-data/output/` first.
- The methylation BED file name pattern may need adjustment depending on how `aligned_bam_to_cpg_scores` is configured.